In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from joblib import Parallel, delayed
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import warnings
import torch.nn.functional as F
from skrebate import ReliefF

import os

# Additional Imports for Hyperparameter Tuning
import optuna
from imblearn.pipeline import Pipeline as ImbPipeline

# Ensure reproducibility
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# Limit each parallel process to one thread per library
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['TORCH_NUM_THREADS'] = '1'  # For PyTorch

warnings.filterwarnings("ignore")  # Optional: Suppress warnings for cleaner output

# ----------------------------
# Data Preparation
# ----------------------------

# Load the dataset
data = pd.read_excel("class123_dataset.xlsx")

# Define feature groups as per user
feature_groups = {
    'Genotype': [
        'rs11225395', 'rs1144393', 'rs650108', 'rs591058', 'rs2252070', 'rs4986938', 'rs1800012', 'rs4789932', 'rs9340799', 'rs970547', 
        'rs1800795', 'rs13946', 'rs12722', 'class1_SNP_risk_score', 'rs7528684', 'rs4919510', 'rs1937810', 'rs6481512', 'rs1249269', 
        'rs12574452', 'rs12429486', 'rs4454832', 'rs2761884', 'rs62051384', 'rs4362400', 'rs2586488', 'rs2277698', 'rs1045485', 
        'rs143383', 'rs17576', 'rs2305948', 'rs1011814', 'rs11154027', 'rs2234693', 'rs1643821', 'rs2010963', 'rs10263021', 'rs149047058', 
        'rs420257', 'rs42517', 'rs42522', 'rs42531', 'rs413826', 'rs2104772', 'rs1330363', 'class12_SNP_risk_score', 'rs3753841', 
        'rs57104447', 'rs1887632', 'rs4654760', 'rs1137101', 'rs2306033', 'rs2277268', 'rs4988321', 'rs11232681', 'rs1718119', 'rs3751143', 
        'rs1544410', 'rs2228570', 'rs4328262', 'rs1021188', 'rs74544784', 'rs78391032', 'rs77569527', 'rs117544024', 'rs912336', 
        'rs3218791', 'rs911263', 'rs2525504', 'rs17756404', 'rs4903399', 'rs10132091', 'rs17583842', 'rs1676303', 'rs11629171', 
        'rs2281518', 'rs2285053', 'rs71404070', 'rs710079', 'rs2858056', 'rs820218', 'rs3018362', 'rs1800470', 'rs1800469', 'rs25487', 
        'rs25489', 'rs2289360', 'rs183364169', 'rs11177', 'rs6617', 'rs3219008', 'rs13107325', 'rs60713544', 'rs145648292', 'rs4244032', 
        'rs12656106', 'rs3045', 'rs187483', 'rs4701616', 'rs144414988', 'rs1800629', 'rs10484958', 'rs4730153', 'rs1800797', 'rs1554606', 
        'rs2237352', 'rs4725069', 'rs12154667', 'rs1548456', 'rs3216902', 'rs35360670', 'rs13317', 'rs1800972', 'rs7035322', 'rs7021589', 
        'rs72758637', 'rs10759753', 'rs3789870', 'rs1138545', 'rs3196378', 'rs1134170', 'rs10992075', 'rs1590', 'rs144371252', 
        'rs761804508', 'class123_SNP_risk_score', 'sex'
    ],
    'History': [
        'Age', 'lower_limb_days_total', 'average_run_hours', 'average_interval_training_frequency', 'EDEQ_total', 'tracking_period_injury',
        'past_stress_injury', 'LEAF-Q', 'Athlete_Score', 'average_run_frequency', 'past_month_injury'
    ],
    'Phenotype': [
        'hip_abduction_peak_torque', 'total_ad_ab_ratio', 'knee_extension_peak_torque', 'knee_flexion_peak_torque', 'navicular_drop', 
        'navicular_drop_asymmetry', 'Q_angle', 'Q_angle_asymmetry', 'VALR_12', 'Impact_peak_12', 'Duty_factor_12', 'BMI', 'BMD_spine',
        'hip_abduction_peak_torque_asymmetry', 'hip_adduction_peak_torque', 'hip_adduction_peak_torque_asymmetry', 
        'knee_extension_peak_torque_asymmetry', 'knee_flexion_peak_torque_asymmetry', 'total_fl_ex_ratio', 'leg_lean_mass', 
        'hip_abduction_peak_angle', 'hip_abduction_peak_angle_asymmetry', 'hip_adduction_peak_angle', 'hip_adduction_peak_angle_asymmetry', 
        'ad_ab_ratio_asymmetry', 'knee_extension_peak_angle', 'knee_extension_peak_angle_asymmetry', 'knee_flexion_peak_angle', 
        'knee_flexion_peak_angle_asymmetry', 'fl_ex_ratio_asymmetry', 'VILR_10', 'VALR_10', 'VILR_asymmetry_10', 'VALR_asymmetry_10', 
        'Impact_peak_10', 'Impact_peak_asymmetry_10', 'Flight_time_10', 'Contact_time_10', 'Duty_factor_10', 'Step_frequency_10', 
        'Cadence_asymmetry_10', 'Duty_factor_asymmetry_10', 'VILR_12', 'VILR_asymmetry_12', 'VALR_asymmetry_12', 
        'Impact_peak_asymmetry_12', 'Flight_time_12', 'Contact_time_12', 'Step_frequency_12', 'Cadence_asymmetry_12', 
        'Duty_factor_asymmetry_12', 'Alt_strike', 'height', 'Mass', 'thigh_lean_mass', 'thigh_ffmi', 'lower_leg_lean_mass', 
        'lower_leg_ffmi', 'leg_ffmi', 'total_lean_mass', 'total_ffmi', 'calf_size', 'BMD_hip', 'BMD_body',
    ],
    'Behaviour': [
        'fat_intake_avg', 'past_month_distance', 'past_month_ratio', 'SC_past_season', 'non_running_past_season', 'fat_intake_BW', 
        'fat_percentage_avg', 'average_energy_availability', 'protein_intake_BW', 'omega3_intake_BW', 'vitaminD_intake_BW', 
        'vitaminC_intake_BW', 'vitaminE_intake_BW', 'calcium_intake_BW', 'copper_intake_BW', 'iron_intake_BW', 'glycine_intake_BW', 
        'arginine_intake_BW', 'past_month_min', 'past_week_ratio', 'past_month_volume_low', 'past_week_ratio_low', 'past_month_ratio_low', 
        'past_month_volume_moderate', 'past_week_ratio_moderate', 'past_month_ratio_moderate', 'past_month_volume_high', 
        'past_week_ratio_high', 'past_month_ratio_high', 'past_month_volume_very_high', 'past_week_ratio_very_high', 
        'past_month_ratio_very_high', 'past_month_calculated_volume', 'past_week_ratio_calculated_volume', 
        'past_month_ratio_calculated_volume', 'resistance_training_past_month', 'resistance_training_past_season', 
        'bodyweight_exercises_past_month', 'bodyweight_exercises_past_season', 'core_stability_past_month', 'core_stability_past_season', 
        'balance_training_past_month', 'balance_training_past_season', 'plyometrics_past_month', 'plyometrics_past_season', 
        'drills_past_month', 'drills_past_season', 'circuit_training_past_month', 'circuit_training_past_season', 'barefoot_past_month', 
        'barefoot_past_season', 'stretching_past_month', 'stretching_past_season', 'SC_past_month', 'non_running_past_month'
    ]
}

# Define predictors and outcome
X = data.drop(columns=['RRI'])  # Predictors
y = data['RRI']  # Outcome

# Ensure that the feature groups exist in the dataset
for group in feature_groups:
    feature_groups[group] = [feature for feature in feature_groups[group] if feature in X.columns]

# Convert X and y to NumPy arrays for ReliefF
X_np = X.values
y_np = y.values

# Initialize ReliefF
relief = ReliefF(n_neighbors=100, n_jobs=-1)
relief.fit(X_np, y_np)

# Get feature scores
feature_scores = pd.Series(relief.feature_importances_, index=X.columns)

# Rank features based on Relief scores (descending)
ranked_features = feature_scores.sort_values(ascending=False)

# Get the list of ranked feature names
selected_features = ranked_features.index.tolist()  # Now a list of column names

print(selected_features)
print(f"Selected Features ({len(selected_features)}):")
print(ranked_features.loc[selected_features])

# Update feature groups based on selected features, preserving ReliefF ranking
selected_feature_groups = {group: [] for group in feature_groups}

for feature in selected_features:
    for group in feature_groups:
        if feature in feature_groups[group]:
            selected_feature_groups[group].append(feature)
            break  # Assuming each feature belongs to only one group

print("Selected Feature Groups:")
for group, features in selected_feature_groups.items():
    print(f"{group} ({len(features)}): {features}")

# Now, redefine X based on selected features
X_selected = X[selected_features].copy()

# Split feature groups
X_genotype = X_selected[selected_feature_groups['Genotype']].values.astype(np.float32)
X_history = X_selected[selected_feature_groups['History']].values.astype(np.float32)
X_phenotype = X_selected[selected_feature_groups['Phenotype']].values.astype(np.float32)
X_behaviour = X_selected[selected_feature_groups['Behaviour']].values.astype(np.float32)

# Convert target to tensor
y = y.values.astype(np.float32)

# ----------------------------
# Dataset and DataLoader
# ----------------------------

class CustomDataset(Dataset):
    def __init__(self, genotype, history, phenotype, behaviour, labels):
        self.genotype = genotype
        self.history = history
        self.phenotype = phenotype
        self.behaviour = behaviour
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.genotype[idx],
            self.history[idx],
            self.phenotype[idx],
            self.behaviour[idx],
            self.labels[idx]
        )

# ----------------------------
# FeatureAttention and MaskedLinear Layers
# ----------------------------

class FeatureAttention(nn.Module):
    def __init__(self, feature_dim):
        super(FeatureAttention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, max(1, feature_dim // 2)),
            nn.ReLU(),
            nn.Linear(max(1, feature_dim // 2), feature_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        weights = self.attention(x)
        return x * weights

class MaskedLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super(MaskedLinear, self).__init__()
        # Initialize the linear layer
        self.linear = nn.Linear(in_features, out_features, bias)
        # Initialize mask parameters with the same shape as weights
        self.mask_param = nn.Parameter(torch.ones_like(self.linear.weight))
        if bias:
            self.bias = self.linear.bias
        else:
            self.register_parameter('bias', None)
    
    def forward(self, x):
        # Apply sigmoid to mask parameters to get gating probabilities
        mask = torch.sigmoid(self.mask_param)
        # Apply the mask to the weights
        masked_weight = self.linear.weight * mask
        return F.linear(x, masked_weight, self.bias)
    
    def get_binary_mask(self, threshold=0.5):
        """
        Returns a binary mask based on the gating parameters and a specified threshold.
        """
        with torch.no_grad():
            mask = torch.sigmoid(self.mask_param)
            binary_mask = (mask > threshold).float()
        return binary_mask

# ----------------------------
# Modified Model Definition with Optional Attention and Mask
# ----------------------------

class CustomMLPWithOptionalComponents(nn.Module):
    def __init__(self, genotype_size, history_size, phenotype_size, behaviour_size,
                 use_attention=True, use_mask=True):
        super(CustomMLPWithOptionalComponents, self).__init__()
        
        self.use_attention = use_attention
        self.use_mask = use_mask

        # Attention for Genotype Features
        if self.use_attention:
            self.attention_genotype = FeatureAttention(genotype_size)
        
        # Define the main layers with MaskedLinear or Linear based on use_mask
        LinearLayer = MaskedLinear if self.use_mask else nn.Linear

        # Genotype to History
        self.genotype_to_history = LinearLayer(genotype_size, history_size, bias=False)
        self.bn_genotype_to_history = nn.BatchNorm1d(history_size)
        self.genotype_to_history_bias = nn.Parameter(torch.zeros(history_size))
        if self.use_attention:
            self.attention_history = FeatureAttention(history_size + 1)  # +1 for extra input
        
        # History to Phenotype
        self.history_to_phenotype = LinearLayer(history_size + 1, phenotype_size, bias=False)
        self.bn_history_to_phenotype = nn.BatchNorm1d(phenotype_size)
        self.history_to_phenotype_bias = nn.Parameter(torch.zeros(phenotype_size))
        if self.use_attention:
            self.attention_phenotype = FeatureAttention(phenotype_size + 1)
        
        # Phenotype to Behaviour
        self.phenotype_to_behaviour = LinearLayer(phenotype_size + 1, behaviour_size, bias=False)
        self.bn_phenotype_to_behaviour = nn.BatchNorm1d(behaviour_size)
        self.phenotype_to_behaviour_bias = nn.Parameter(torch.zeros(behaviour_size))
        if self.use_attention:
            self.attention_behaviour = FeatureAttention(behaviour_size + 1)
        
        # Behaviour to Output
        self.behaviour_to_output = LinearLayer(behaviour_size + 1, 1)
        
        # Extra regular node layers (no batch norm needed)
        self.genotype_to_history_extra = LinearLayer(genotype_size, 1, bias=True)
        self.history_to_phenotype_extra = LinearLayer(history_size + 1, 1, bias=True)
        self.phenotype_to_behaviour_extra = LinearLayer(phenotype_size + 1, 1, bias=True)
        
        # Activation function
        self.relu = torch.nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
        # Initialize weights using Xavier/Glorot initialization
        for layer in [
            self.genotype_to_history, 
            self.history_to_phenotype, 
            self.phenotype_to_behaviour, 
            self.behaviour_to_output,
            self.genotype_to_history_extra,
            self.history_to_phenotype_extra,
            self.phenotype_to_behaviour_extra
        ]:
            if self.use_mask:
                nn.init.xavier_uniform_(layer.linear.weight)
                if layer.linear.bias is not None:
                    nn.init.zeros_(layer.linear.bias)
            else:
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
    
    def forward(self, genotype, history, phenotype, behaviour):
        # Genotype to History with optional attention
        if self.use_attention:
            genotype_att = self.attention_genotype(genotype)
        else:
            genotype_att = genotype
        main_history_input = self.genotype_to_history(genotype_att) * history
        main_history_input = self.bn_genotype_to_history(main_history_input)
        main_history_input = main_history_input + self.genotype_to_history_bias
        
        # Extra input (no batch norm)
        extra_history_input = self.genotype_to_history_extra(genotype_att)
        combined_history_input = torch.cat([main_history_input, extra_history_input], dim=1)
        history_output = self.relu(combined_history_input)
        
        # History to Phenotype with optional attention
        if self.use_attention:
            history_att = self.attention_history(history_output)
        else:
            history_att = history_output
        main_phenotype_input = self.history_to_phenotype(history_att) * phenotype
        main_phenotype_input = self.bn_history_to_phenotype(main_phenotype_input)
        main_phenotype_input = main_phenotype_input + self.history_to_phenotype_bias
        
        # Extra input for phenotype (no batch norm)
        extra_phenotype_input = self.history_to_phenotype_extra(history_att)
        combined_phenotype_input = torch.cat([main_phenotype_input, extra_phenotype_input], dim=1)
        phenotype_output = self.relu(combined_phenotype_input) 
        
        # Phenotype to Behaviour with optional attention
        if self.use_attention:
            phenotype_att = self.attention_phenotype(phenotype_output)
        else:
            phenotype_att = phenotype_output
        main_behaviour_input = self.phenotype_to_behaviour(phenotype_att) * behaviour
        main_behaviour_input = self.bn_phenotype_to_behaviour(main_behaviour_input)
        main_behaviour_input = main_behaviour_input + self.phenotype_to_behaviour_bias

        # Extra input for behaviour (no batch norm)
        extra_behaviour_input = self.phenotype_to_behaviour_extra(phenotype_att)
        combined_behaviour_input = torch.cat([main_behaviour_input, extra_behaviour_input], dim=1)
        behaviour_output = self.relu(combined_behaviour_input) 
        
        # Behaviour to Output with optional attention
        if self.use_attention:
            behaviour_att = self.attention_behaviour(behaviour_output)
        else:
            behaviour_att = behaviour_output
        final_input = self.behaviour_to_output(behaviour_att)
        final_output = self.sigmoid(final_input)
        
        return final_output.squeeze()  # Return as (batch_size,)

# ----------------------------
# Hyperparameter Tuning with Optuna
# ----------------------------

# Define the objective function
def objective(trial):
    # Hyperparameters to tune
    # Number of features per group
    n_genotype = trial.suggest_int('n_genotype', 1, len(selected_feature_groups['Genotype']))
    n_history = trial.suggest_int('n_history', 1, len(selected_feature_groups['History']))
    n_phenotype = trial.suggest_int('n_phenotype', 1, len(selected_feature_groups['Phenotype']))
    n_behaviour = trial.suggest_int('n_behaviour', 1, len(selected_feature_groups['Behaviour']))
    
    # Learning rate
    lr = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    
    # Number of epochs
    epochs = trial.suggest_int('epochs', 500, 3000)
    
    # Batch size
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128, 256, 512])
    
    # Whether to use attention and mask
    use_attention = True
    use_mask = False
    
    # Select features based on the number of features per group
    selected_genotype_features = selected_feature_groups['Genotype'][:n_genotype]
    selected_history_features = selected_feature_groups['History'][:n_history]
    selected_phenotype_features = selected_feature_groups['Phenotype'][:n_phenotype]
    selected_behaviour_features = selected_feature_groups['Behaviour'][:n_behaviour]
    
    # Combine selected features
    current_selected_features = selected_genotype_features + selected_history_features + \
                                 selected_phenotype_features + selected_behaviour_features

    print(current_selected_features)
    
    # Prepare data based on current_selected_features
    X_current = X[current_selected_features].copy()
    
    # Update feature groups
    current_feature_groups = {
        'Genotype': selected_genotype_features,
        'History': selected_history_features,
        'Phenotype': selected_phenotype_features,
        'Behaviour': selected_behaviour_features
    }
    
    # Split feature groups
    X_genotype_current = X_current[current_feature_groups['Genotype']].values.astype(np.float32)
    X_history_current = X_current[current_feature_groups['History']].values.astype(np.float32)
    X_phenotype_current = X_current[current_feature_groups['Phenotype']].values.astype(np.float32)
    X_behaviour_current = X_current[current_feature_groups['Behaviour']].values.astype(np.float32)
    
    # Convert target to tensor
    y_current = y  # Already a NumPy array
    
    # Stratified K-Fold Cross Validation with 10 folds
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    
    auc_scores = []
    
    # Define the fold processing function
    def train_evaluate_fold(train_index, valid_index):
        # Split the data
        X_train_gen, X_valid_gen = X_genotype_current[train_index], X_genotype_current[valid_index]
        X_train_hist, X_valid_hist = X_history_current[train_index], X_history_current[valid_index]
        X_train_pheno, X_valid_pheno = X_phenotype_current[train_index], X_phenotype_current[valid_index]
        X_train_behav, X_valid_behav = X_behaviour_current[train_index], X_behaviour_current[valid_index]
        y_train_fold, y_valid_fold = y_current[train_index], y_current[valid_index]
        
        # Handle class imbalance using SMOTE (you can choose other methods)
        X_train_combined = np.hstack((X_train_gen, X_train_hist, X_train_pheno, X_train_behav))
        X_train_res, y_train_res = (X_train_combined, y_train_fold)  # Placeholder for SMOTE
        
        # After resampling, split back into feature groups
        n_gen = X_train_gen.shape[1]
        n_hist = X_train_hist.shape[1]
        n_pheno = X_train_pheno.shape[1]
        n_behav = X_train_behav.shape[1]
        
        X_train_gen_res = X_train_res[:, :n_gen]
        X_train_hist_res = X_train_res[:, n_gen:n_gen+n_hist]
        X_train_pheno_res = X_train_res[:, n_gen+n_hist:n_gen+n_hist+n_pheno]
        X_train_behav_res = X_train_res[:, n_gen+n_hist+n_pheno:]
        
        # Create datasets and dataloaders
        train_dataset = CustomDataset(
            genotype=X_train_gen_res,
            history=X_train_hist_res,
            phenotype=X_train_pheno_res,
            behaviour=X_train_behav_res,
            labels=y_train_res
        )
        
        valid_dataset = CustomDataset(
            genotype=X_valid_gen,
            history=X_valid_hist,
            phenotype=X_valid_pheno,
            behaviour=X_valid_behav,
            labels=y_valid_fold
        )
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
        
        # Initialize the model
        model = CustomMLPWithOptionalComponents(
            genotype_size=X_train_gen_res.shape[1],
            history_size=X_train_hist_res.shape[1],
            phenotype_size=X_train_pheno_res.shape[1],
            behaviour_size=X_train_behav_res.shape[1],
            use_attention=use_attention,
            use_mask=use_mask
        ).to(device)
        
        # Define optimizer and loss function
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.BCELoss()
        
        # Training loop
        model.train()
        for epoch in range(epochs):
            for batch in train_loader:
                genotype, history, phenotype, behaviour, labels = batch
                genotype = genotype.to(device)
                history = history.to(device)
                phenotype = phenotype.to(device)
                behaviour = behaviour.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                outputs = model(genotype, history, phenotype, behaviour)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
        
        # Evaluation
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for batch in valid_loader:
                genotype, history, phenotype, behaviour, labels = batch
                genotype = genotype.to(device)
                history = history.to(device)
                phenotype = phenotype.to(device)
                behaviour = behaviour.to(device)
                
                outputs = model(genotype, history, phenotype, behaviour)
                all_preds.extend(outputs.cpu().numpy())
                all_labels.extend(labels.numpy())
        
        # Compute ROC AUC
        auc = roc_auc_score(all_labels, all_preds)
        return auc
    
    # Parallelize the fold processing
    results = Parallel(n_jobs=10)(
        delayed(train_evaluate_fold)(train_idx, valid_idx) for train_idx, valid_idx in skf.split(X_current, y_current)
    )
    
    auc_scores = results  # List of AUCs from each fold
    
    # Return the average AUC across folds
    return np.mean(auc_scores)

# Set device
device = torch.device('cpu')

# Create the Optuna study
study = optuna.create_study(direction='maximize')

# Optimize
study.optimize(objective, n_trials=500, timeout=None)  # Adjust n_trials and timeout as needed

# Print the best trial
print("Best Trial:")
trial = study.best_trial

print(f"  AUC: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2024-11-04 04:08:09,870] A new study created in memory with name: no-name-cb0a12f7-c71a-4699-b34a-30767603aedf


['rs591058', 'rs1800797', 'rs13946', 'rs25487', 'rs2228570', 'rs12722', 'rs2104772', 'rs1144393', 'rs3196378', 'rs1544410', 'rs2237352', 'rs1137101', 'rs10263021', 'rs4454832', 'rs7528684', 'rs2234693', 'rs10132091', 'rs7035322', 'rs1330363', 'rs1011814', 'rs1249269', 'rs4725069', 'rs820218', 'rs62051384', 'rs11232681', 'rs4701616', 'rs1800629', 'rs11177', 'rs143383', 'rs6617', 'rs6481512', 'rs17756404', 'rs4986938', 'rs13317', 'rs4730153', 'rs911263', 'rs970547', 'rs11225395', 'rs3018362', 'rs11629171', 'rs10992075', 'rs4789932', 'rs2252070', 'rs3789870', 'rs1590', 'rs42531', 'rs11154027', 'rs10759753', 'rs2761884', 'rs4362400', 'Q_angle', 'rs3219008', 'hip_abduction_peak_torque_asymmetry', 'rs3218791', 'total_ad_ab_ratio', 'rs17576', 'rs2289360', 'Age', 'rs1134170', 'rs2525504', 'fat_intake_BW', 'rs4919510', 'tracking_period_injury', 'rs1937810', 'Step_frequency_10', 'Athlete_Score', 'rs1718119', 'rs10484958', 'rs2285053', 'thigh_ffmi', 'class12_SNP_risk_score', 'fat_percentage_avg',

[I 2024-11-04 04:22:27,907] Trial 0 finished with value: 0.6803103554177435 and parameters: {'n_genotype': 98, 'n_history': 7, 'n_phenotype': 18, 'n_behaviour': 20, 'learning_rate': 0.008443987362225248, 'epochs': 2763, 'batch_size': 128}. Best is trial 0 with value: 0.6803103554177435.


['rs591058', 'rs1800797', 'rs13946', 'rs25487', 'rs2228570', 'rs12722', 'rs2104772', 'rs1144393', 'rs3196378', 'rs1544410', 'rs2237352', 'rs1137101', 'rs10263021', 'rs4454832', 'rs7528684', 'rs2234693', 'rs10132091', 'rs7035322', 'rs1330363', 'rs1011814', 'rs1249269', 'rs4725069', 'rs820218', 'rs62051384', 'rs11232681', 'rs4701616', 'rs1800629', 'rs11177', 'rs143383', 'rs6617', 'rs6481512', 'rs17756404', 'rs4986938', 'rs13317', 'rs4730153', 'rs911263', 'rs970547', 'rs11225395', 'rs3018362', 'rs11629171', 'rs10992075', 'rs4789932', 'rs2252070', 'rs3789870', 'rs1590', 'rs42531', 'rs11154027', 'rs10759753', 'rs2761884', 'rs4362400', 'rs3219008', 'rs3218791', 'rs17576', 'rs2289360', 'rs1134170', 'rs2525504', 'rs4919510', 'rs1937810', 'rs1718119', 'rs10484958', 'rs2285053', 'class12_SNP_risk_score', 'rs1800972', 'rs4328262', 'rs1800469', 'rs2306033', 'rs9340799', 'rs17583842', 'rs4244032', 'rs1643821', 'rs1800470', 'rs3045', 'rs4903399', 'rs3753841', 'rs2277698', 'rs1800012', 'rs12656106', 

[I 2024-11-04 04:30:48,043] Trial 1 finished with value: 0.7159134722554701 and parameters: {'n_genotype': 91, 'n_history': 3, 'n_phenotype': 39, 'n_behaviour': 28, 'learning_rate': 9.168296170297456e-05, 'epochs': 2643, 'batch_size': 256}. Best is trial 1 with value: 0.7159134722554701.


...

['rs591058', 'rs1800797', 'rs13946', 'rs25487', 'rs2228570', 'rs12722', 'rs2104772', 'rs1144393', 'rs3196378', 'rs1544410', 'rs2237352', 'rs1137101', 'rs10263021', 'rs4454832', 'rs7528684', 'rs2234693', 'rs10132091', 'rs7035322', 'rs1330363', 'rs1011814', 'rs1249269', 'rs4725069', 'rs820218', 'rs62051384', 'rs11232681', 'rs4701616', 'rs1800629', 'rs11177', 'rs143383', 'rs6617', 'rs6481512', 'rs17756404', 'rs4986938', 'rs13317', 'rs4730153', 'rs911263', 'rs970547', 'rs11225395', 'rs3018362', 'rs11629171', 'rs10992075', 'rs4789932', 'rs2252070', 'rs3789870', 'rs1590', 'rs42531', 'rs11154027', 'rs10759753', 'rs2761884', 'rs4362400', 'rs3219008', 'rs3218791', 'rs17576', 'rs2289360', 'rs1134170', 'rs2525504', 'Age', 'Q_angle', 'hip_abduction_peak_torque_asymmetry', 'total_ad_ab_ratio', 'Step_frequency_10', 'thigh_ffmi', 'hip_adduction_peak_torque_asymmetry', 'BMD_hip', 'calf_size', 'Flight_time_12', 'total_fl_ex_ratio', 'navicular_drop', 'Step_frequency_12', 'knee_extension_peak_angle_asymm

[I 2024-11-08 14:48:27,641] Trial 489 finished with value: 0.7212373066255567 and parameters: {'n_genotype': 56, 'n_history': 1, 'n_phenotype': 52, 'n_behaviour': 12, 'learning_rate': 0.001437256004360576, 'epochs': 1576, 'batch_size': 256}. Best is trial 211 with value: 0.7531640655561265.


['rs591058', 'rs1800797', 'rs13946', 'rs25487', 'rs2228570', 'rs12722', 'rs2104772', 'rs1144393', 'rs3196378', 'rs1544410', 'rs2237352', 'rs1137101', 'rs10263021', 'rs4454832', 'rs7528684', 'rs2234693', 'rs10132091', 'rs7035322', 'rs1330363', 'rs1011814', 'rs1249269', 'rs4725069', 'rs820218', 'rs62051384', 'rs11232681', 'rs4701616', 'rs1800629', 'rs11177', 'rs143383', 'rs6617', 'rs6481512', 'rs17756404', 'rs4986938', 'rs13317', 'rs4730153', 'rs911263', 'rs970547', 'rs11225395', 'rs3018362', 'rs11629171', 'rs10992075', 'rs4789932', 'rs2252070', 'rs3789870', 'rs1590', 'rs42531', 'rs11154027', 'rs10759753', 'rs2761884', 'rs4362400', 'rs3219008', 'Age', 'Q_angle', 'hip_abduction_peak_torque_asymmetry', 'total_ad_ab_ratio', 'Step_frequency_10', 'thigh_ffmi', 'hip_adduction_peak_torque_asymmetry', 'BMD_hip', 'calf_size', 'Flight_time_12', 'total_fl_ex_ratio', 'navicular_drop', 'Step_frequency_12', 'knee_extension_peak_angle_asymmetry', 'VALR_10', 'VALR_12', 'Q_angle_asymmetry', 'VILR_12', 'V

[I 2024-11-08 14:54:47,088] Trial 490 finished with value: 0.6976906186356555 and parameters: {'n_genotype': 51, 'n_history': 1, 'n_phenotype': 51, 'n_behaviour': 5, 'learning_rate': 4.824070242956016e-05, 'epochs': 1623, 'batch_size': 256}. Best is trial 211 with value: 0.7531640655561265.


['rs591058', 'rs1800797', 'rs13946', 'rs25487', 'rs2228570', 'rs12722', 'rs2104772', 'rs1144393', 'rs3196378', 'rs1544410', 'rs2237352', 'rs1137101', 'rs10263021', 'rs4454832', 'rs7528684', 'rs2234693', 'rs10132091', 'rs7035322', 'rs1330363', 'rs1011814', 'rs1249269', 'rs4725069', 'rs820218', 'rs62051384', 'rs11232681', 'rs4701616', 'rs1800629', 'rs11177', 'rs143383', 'rs6617', 'rs6481512', 'rs17756404', 'rs4986938', 'rs13317', 'rs4730153', 'rs911263', 'rs970547', 'rs11225395', 'rs3018362', 'rs11629171', 'rs10992075', 'rs4789932', 'rs2252070', 'rs3789870', 'rs1590', 'rs42531', 'rs11154027', 'Age', 'Q_angle', 'hip_abduction_peak_torque_asymmetry', 'total_ad_ab_ratio', 'Step_frequency_10', 'thigh_ffmi', 'hip_adduction_peak_torque_asymmetry', 'BMD_hip', 'calf_size', 'Flight_time_12', 'total_fl_ex_ratio', 'navicular_drop', 'Step_frequency_12', 'knee_extension_peak_angle_asymmetry', 'VALR_10', 'VALR_12', 'Q_angle_asymmetry', 'VILR_12', 'VILR_10', 'hip_adduction_peak_torque', 'thigh_lean_mas